# Random EINS Data Generation Script

This juoyter notebook seeks to expand on the current volume of EINS dataset by randomly generating new data given an adjusted set of parameters

Importing the libraries

In [1]:
import datetime
import pandas as pd
import re
import json
import itertools
import random
import string
import os
import joblib
import sys

sys.path.append('../../../Litigation Clustering/')
sys.path.append('../../../Ssic Mapping/')

from ssic_mapping import ssic_mapping
from litigation_clustering import label_litigation_main_cause, label_litigation_case_status, label_litigation_party_status

Runner Configuration

In [2]:
with open('../../eins_variables.json') as f:
    data = json.load(f)

In [3]:
def randdate(start_date, end_date, update=True):
    time_between_dates = end_date - start_date
    days_between_dates = time_between_dates.days
    random_number_of_days = random.randrange(days_between_dates)
    random_date = start_date + datetime.timedelta(days=random_number_of_days)
    if update:
        global randomDate
        randomDate = random_date
    return random_date

def randdates(num, start_date, end_date, update=True):
    random_dates = []
    for _ in range(num):
        random_date = randdate(start_date, end_date, update=False)
        random_dates.append(random_date)
    if update:
        global randomDates
        randomDates = random_dates
    return random_dates

In [4]:
# Number of random datasets to generate
dataCount = 10

# Intialise date variable
global randomDate
global randomDates
req_date = randdate(datetime.datetime(2023, 1, 1), datetime.datetime.today())
reg_date = randdate(datetime.datetime(2000, 1, 1), req_date)

# Imported fields
MAIN_CAUSE = list(pd.read_csv('../../../Dataset/litigation_data/litigation_main_cause.csv')['Main Cause'])
CASE_STATUS = list(pd.read_csv('../../../Dataset/litigation_data/litigation_case_status.csv')['Case Status'])
PARTY_STATUS = list(pd.read_csv('../../../Dataset/litigation_data/litigation_party_status.csv')['Party (Plaintiff/Defendant) Status'])
ENTITY_TYPE_INDICATOR = list(pd.read_csv('../../../ibs/uen mapping/UEN Mapping.csv')['Entity-Type Indicator'])
CHARGE_STATUS = data['CHARGE_STATUS']
COUNTRY = data['COUNTRY']
SHARE_TYPE = data['SHARE_TYPE']
COY_TYPE = data['COY_TYPE']
COY_STATUS = data['COY_STATUS']
AUDITOR = data['AUDITOR']
NATIONALITY = data['NATIONALITY']
OFFICERS_POS_HELD = data['OFFICERS_POS_HELD']
WRIT_SUM_COURT_TYPE = data['WRIT_SUM_COURT_TYPE']
FINANCIAL_REL_SEARCH_COUNT_VAL = data['FINANCIAL_REL_SEARCH_COUNT_VAL']
COMMERCIAL_REL_SEARCH_COUNT_VAL = data['COMMERCIAL_REL_SEARCH_COUNT_VAL']
LEGAL_REL_SEARCH_COUNT_VAL = data['LEGAL_REL_SEARCH_COUNT_VAL']
firstName = data['firstName']
middleName = data['middleName']
sector = data['sector']
streetNameFirst = data['streetNameFirst']
streetNameLast = data['streetNameLast']
companyType = data['companyType']
humanFirstName = data['humanFirstName']
humanMiddleName = data['humanMiddleName']
principleActivities = data['principleActivities']

# Duplicated fields (Singular / Composite)
ENTITY_ID = random.choice([
    f'{"".join(random.choices(string.digits, k=8))}{random.choice(string.ascii_uppercase)}', 
    f'{reg_date.year}{"".join(random.choices(string.digits, k=5))}{random.choice(string.ascii_uppercase)}', 
    f'T{reg_date.year % 100}{random.choice(ENTITY_TYPE_INDICATOR)}{"".join(random.choices(string.digits, k=4))}{random.choice(string.ascii_uppercase)}', 
])
ENTITY_NAME = f'{random.choice(firstName)} {random.choice(middleName)} {random.choice(sector)} {random.choice(companyType)}'

# Multiple Occurence fields (variable according to each report)
NUM_OF_CHARGES = random.randint(1, 5)
NUM_OF_SHAREHOLDERS = random.randint(1, 5)
NUM_OF_OFFICERS_AUTH_REP = random.randint(1, 8)
WRIT_SUM_ORGS_CLAIM_COUNT = random.randint(1, 3)
WINDING_UP_COUNT = random.randint(1, 5)
OSOAOPOM_COUNT = random.randint(1, 3)
ACRRED_COUNT = random.randint(1, 5)

# Predefined-fields
CURR_FYE_REV = random.randint(10000000000, 90000000000)
PAT_CURR_FYE = CURR_FYE_REV - random.randint(100000000, 999000000)
SHAREHOLDERSFUND_CURR_FYE = CURR_FYE_REV - random.randint(900000000, 999000000)
PREV_FYE_REV = random.randint(10000000000, 90000000000)
PAT_PREV_FYE = PREV_FYE_REV - random.randint(100000000, 999000000)
SHAREHOLDERSFUND_PREV_FYE = PREV_FYE_REV - random.randint(900000000, 999000000)

# Initialise ssic mapping
ssic_mapper = pd.read_csv('../../../Dataset/ssic_data/ssic_mapping.csv', dtype=str)

# Initialise clustering models
litigation_main_cause_clustering_model = joblib.load('../../../Litigation Clustering/litigation_main_cause_clustering.pkl')
litigation_case_status_clustering_model = joblib.load('../../../Litigation Clustering/litigation_case_status_clustering.pkl')
litigation_party_status_clustering_model = joblib.load('../../../Litigation Clustering/litigation_party_status_clustering.pkl')

# Limits for each random attribute generated
random_params = {
    'Request Criteria': {
        'request_date':                                         req_date.strftime('%Y/%m/%d'),
        'requestor_name':                                       f'{random.choice(firstName)} {random.choice(middleName)} {random.choice(sector)} {random.choice(companyType)}', 
        'requested_entity_name':                                ENTITY_NAME,
        'requested_entity_id':                                  ENTITY_ID,
        'request_id':                                           'EINSG SAMPLE', 
    },
    'Search Record':{
        'entity_name':                                          ENTITY_NAME,
        'entity_id':                                            ENTITY_ID, 
    },
    'Details of Company':{
        'entity_name':                                          ENTITY_NAME,
        'entity_id':                                            ENTITY_ID,
        'registration_date':                                    reg_date.strftime('%Y/%m/%d'),
        'country_or_region_of_incorporation_or_registration':   random.choice(COUNTRY),
        'date_of_name_change':                                  randdate(reg_date, req_date).strftime('%Y/%m/%d'),
        'former_name':                                          f'{ENTITY_NAME}',
        'effective_date_of_former_name':                        randdate(reg_date, randomDate).strftime('%Y/%m/%d'),
        'company_type':                                         random.choice(COY_TYPE),
        'registered_office_address':                            f'{random.choice(streetNameFirst)} {random.choice(streetNameLast)} #{random.randint(1, 30)}- {random.randint(1, 50)} SINGAPORE {random.randint(100000, 999999)}',
        'date_of_address_change':                               randdate(reg_date, req_date).strftime('%Y/%m/%d'),
        'principal_activities':                                 random.choice(ssic_mapper['Sub-Class'] + ' ('+ ssic_mapper[ssic_mapper.columns[5]] + ')'),
        'status':                                               random.choice(COY_STATUS),
        'status_date':                                          randdate(reg_date, req_date).strftime('%Y/%m/%d'),  
    },
    'Capital Structure':{
        'issued_ordinary_no_of_shares':                         random.randint(10000, 10000000),
        'currency_of_issued_ordinary':                          'SINGAPORE, DOLLARS',
        'amount_of_issued_ordinary':                            random.randint(10000, 10000000),
        'paid_up_ordinary_no_of_shares':                        random.randint(10000, 10000000),
        'currency_of_paid_up_ordinary':                         'SINGAPORE, DOLLARS',
        'amount_of_paid_up_ordinary':                           random.randint(10000, 10000000),
        'paid_up_preference_no_of_shares':                      random.randint(10000, 10000000),
        'currency_of_paid_up_preference':                       'SINGAPORE, DOLLARS',
        'amount_of_paid_up_preference':                         random.randint(10000, 10000000),
        'paid_up_others_no_of_shares':                          random.randint(10000, 10000000), 
        'currency_of_paid_up_others':                           'SINGAPORE, DOLLARS', 
        'amount_of_paid_up_others':                             random.randint(10000, 10000000),
    },
    'Charges':{
        'charge_id':                                            [f'C{i.year}{"".join(random.choices(string.digits, k=5))}' for i in randdates(NUM_OF_CHARGES, reg_date, req_date)],
        'date_of_charge':                                       [randomDates[i].strftime('%Y/%m/%d') for i in range(NUM_OF_CHARGES)],
        'chargee':                                              ['BANK NAME' for _ in range(NUM_OF_CHARGES)],
        'amount_secured_and_currency':                          [f'{random.randint(10000, 100000000)} SINGAPORE, DOLLARS' for _ in range(NUM_OF_CHARGES)],
        'status':                                               [random.choice(CHARGE_STATUS) for i in range(NUM_OF_CHARGES)],
    },
    'Auditors':{
        'auditor_name':                                         random.choice(AUDITOR), 
        'date_of_appointment':                                  randdate(reg_date, req_date).strftime('%Y/%m/%d'),
    },
    'Officers Auth. Rep':{
        'employee_name':                                        [f'{random.choice(humanFirstName)} {random.choice(humanMiddleName)}' for _ in range(NUM_OF_OFFICERS_AUTH_REP)],
        'employee_id':                                          [''.join(random.choices(string.ascii_uppercase + string.digits, k=6)) for _ in range(NUM_OF_OFFICERS_AUTH_REP)],
        'employee_address':                                     [f'{random.choice(streetNameFirst)} {random.choice(streetNameLast)} #{random.randint(1, 30)}- {random.randint(1, 50)} {random.choice(NATIONALITY)} {random.randint(100000, 999999)}' for _ in range(NUM_OF_OFFICERS_AUTH_REP)],
        'date_of_change_of_address':                            [random.choice(['-', i.strftime('%Y/%m/%d')]) for i in randdates(NUM_OF_OFFICERS_AUTH_REP, reg_date, req_date)],
        'nationality_or_citizenship':                           [random.choice(NATIONALITY) for _ in range(NUM_OF_OFFICERS_AUTH_REP)],
        'date_of_appointment':                                  [i.strftime('%Y/%m/%d') for i in randdates(NUM_OF_OFFICERS_AUTH_REP, reg_date, req_date)],
        'position_held':                                        [random.choice(OFFICERS_POS_HELD) for _ in range(NUM_OF_OFFICERS_AUTH_REP)],
        'entity_id':                                            ENTITY_ID,
    },
    'Shareholders':{
        'shareholder_name':                                     [f'{random.choice(firstName)} {random.choice(middleName)} {random.choice(sector)} {random.choice(companyType)}' for _ in range(NUM_OF_SHAREHOLDERS)],
        'shareholder_id':                                       [''.join(random.choices(string.ascii_uppercase + string.digits, k=8)) for _ in range(NUM_OF_SHAREHOLDERS)],
        'nationality_or_citizenship':                           [random.choice(NATIONALITY) for i in range(NUM_OF_SHAREHOLDERS)],
        'shareholder_address':                                  [f'{random.choice(streetNameFirst)} {random.choice(streetNameLast)} #{random.randint(1, 30)}- {random.randint(1, 50)} {random.choice(NATIONALITY)} {random.randint(100000, 999999)}' for _ in range(NUM_OF_SHAREHOLDERS)],
        'date_of_change_of_address':                            [random.choice(['-', i.strftime('%Y/%m/%d')]) for i in randdates(NUM_OF_SHAREHOLDERS, reg_date, req_date)],
        'type_of_shares':                                       [random.choice(SHARE_TYPE) for _ in range(NUM_OF_SHAREHOLDERS)],
        'no_of_shares':                                         [random.randint(10000, 100000000) for _ in range(NUM_OF_SHAREHOLDERS)],
        'currency':                                             ['SINGAPORE, DOLLARS' for _ in range(NUM_OF_SHAREHOLDERS)],
    },
    'Compliance Record':{
        'date_of_last_agm':                                     randdate(reg_date, req_date).strftime('%Y/%m/%d'),
        'date_of_last_ar':                                      randdate(reg_date, req_date).strftime('%Y/%m/%d'),
        'date_of_ac_laid_at_last_agm':                          randdate(reg_date, req_date).strftime('%Y/%m/%d'),
    },
    'Financial Data':{
        'current_fye':                                          f'31/12/{req_date.year - 1}',
        'previous_fye':                                         f'31/12/{req_date.year - 2}',
        'currency':                                             'SINGAPORE, DOLLARS',
        'current_fye_revenue':                                  CURR_FYE_REV,
        'previous_fye_revenue':                                 PREV_FYE_REV,
        'percentage_change_revenue':                            round((CURR_FYE_REV - PREV_FYE_REV) / PREV_FYE_REV * 100, 2),
        'current_fye_profit_after_tax':                         PAT_CURR_FYE,
        'previous_fye_profit_after_tax':                        PAT_PREV_FYE,
        'percentage_change_profit_after_tax':                   round((PAT_CURR_FYE - PAT_PREV_FYE) / PAT_PREV_FYE * 100, 2),
        'current_fye_shareholders_fund':                        SHAREHOLDERSFUND_CURR_FYE,
        'previous_fye_shareholders_fund':                       SHAREHOLDERSFUND_PREV_FYE,
        'percentage_change_shareholders_fund':                  round((SHAREHOLDERSFUND_CURR_FYE - SHAREHOLDERSFUND_PREV_FYE) / SHAREHOLDERSFUND_PREV_FYE * 100, 2),
    },
    'Key Ratios':{
        'current_fye':                                          f'31/12/{req_date.year - 1}',
        'previous_fye':                                         f'31/12/{req_date.year - 2}',
        'current_fye_profit_margin':                            f'{round(random.uniform(2.0, 5.0), 2)}%',
        'previous_fye_profit_margin':                           f'{round(random.uniform(2.0, 5.0), 2)}%',
        'current_fye_liquidity_times':                          round(random.uniform(0.0, 3.0), 2),
        'previous_fye_liquidity_times':                         round(random.uniform(0.0, 3.0), 2),
        'current_fye_debt_times':                               round(random.uniform(0.0, 3.0), 2),
        'previous_fye_debt_times':                              round(random.uniform(0.0, 3.0), 2),
    },
    'Analysis':{
        'fye':                                                  f'{req_date.year - 1}', 
        'company_level_revenue_growth':                         f'{round(random.uniform(5.0, 60.0), 2)}%',
        'industry_median_revenue_growth':                       f'{round(random.uniform(5.0, 30.0), 2)}%',
        'company_level_return_on_assets':                       f'{round(random.uniform(2.0, 5.0), 2)}%',
        'industry_median_return_on_assets':                     f'{round(random.uniform(2.0, 5.0), 2)}%',
        'company_level_total_asset_turnover':                   f'{round(random.uniform(2.0, 30.0), 2)}%',
        'industry_median_total_asset_turnover':                 f'{round(random.uniform(2.0, 15.0), 2)}%',
        'company_level_net_profit_margin':                      f'{round(random.uniform(2.0, 5.0), 2)}%',
        'industry_median_net_profit_margin':                    f'{round(random.uniform(2.0, 5.0), 2)}%',
    },
    'Writ of Sum. Org. Claim':{
        'sn':                                                   [i + 1 for i in range(WRIT_SUM_ORGS_CLAIM_COUNT)],
        'court_type':                                           [random.choice(WRIT_SUM_COURT_TYPE) for _ in range(WRIT_SUM_ORGS_CLAIM_COUNT)],
        'case_no':                                              [random.randint(1, 10000) for _ in range(WRIT_SUM_ORGS_CLAIM_COUNT)],
        'year':                                                 [min(randomDates[i], randomDates[-i - 1]).year for i in range(len(randdates(2 * WRIT_SUM_ORGS_CLAIM_COUNT, reg_date, req_date)) // 2)], 
        'date_of_filling':                                      [min(randomDates[i], randomDates[-i - 1]).strftime('%Y/%m/%d') for i in range(WRIT_SUM_ORGS_CLAIM_COUNT)],
        'plaintiff':                                            [f'{random.choice(firstName)} {random.choice(middleName)} {random.choice(sector)} {random.choice(companyType)}' for _ in range(WRIT_SUM_ORGS_CLAIM_COUNT)],
        'case_status':                                          [random.choice(CASE_STATUS) for _ in range(WRIT_SUM_ORGS_CLAIM_COUNT)],
        'date_of_case_status':                                  [randdate(min(randomDates[i], randomDates[-i - 1]), max(randomDates[i], randomDates[-i - 1])).strftime('%Y/%m/%d') for i in range(WRIT_SUM_ORGS_CLAIM_COUNT)],
        'case_status_remarks':                                  [random.choice(CASE_STATUS) for _ in range(WRIT_SUM_ORGS_CLAIM_COUNT)],
        'plaintiff_status':                                     [random.choice(PARTY_STATUS) for _ in range(WRIT_SUM_ORGS_CLAIM_COUNT)],
        'date_of_plaitiff_status':                              [randdate(min(randomDates[i], randomDates[-i - 1]), max(randomDates[i], randomDates[-i - 1])).strftime('%Y/%m/%d') for i in range(WRIT_SUM_ORGS_CLAIM_COUNT)],
        'defendant_status':                                     [random.choice(PARTY_STATUS) for _ in range(WRIT_SUM_ORGS_CLAIM_COUNT)],
        'date_of_defendant_status':                             [randdate(min(randomDates[i], randomDates[-i - 1]), max(randomDates[i], randomDates[-i - 1])).strftime('%Y/%m/%d') for i in range(WRIT_SUM_ORGS_CLAIM_COUNT)],
        'amount':                                               [f'1) SGD {random.randint(100, 50000)}000' for _ in range(WRIT_SUM_ORGS_CLAIM_COUNT)],
        'last_updated':                                         [randdate(randomDates[i], req_date).strftime('%Y/%m/%d') for i in range(WRIT_SUM_ORGS_CLAIM_COUNT)],
        'main_cause':                                           [random.choice(MAIN_CAUSE) for i in range(WRIT_SUM_ORGS_CLAIM_COUNT)],
    },
    'Winding Up':{
        'type':                                                 ['WINDING UP' for _ in range(WINDING_UP_COUNT)],
        'case_no':                                              [random.randint(1, 10000) for _ in range(WINDING_UP_COUNT)], 
        'year':                                                 [min(randomDates[i], randomDates[-i - 1]).year for i in range(len(randdates(2 * WINDING_UP_COUNT, reg_date, req_date)) // 2)], 
        'date_of_filling':                                      [min(randomDates[i], randomDates[-i - 1]).strftime('%Y/%m/%d') for i in range(WINDING_UP_COUNT)],
        'other_party_name':                                     [f'{random.choice(firstName)} {random.choice(middleName)} {random.choice(sector)} {random.choice(companyType)}' for _ in range(WINDING_UP_COUNT)],
        'case_status':                                          [random.choice(CASE_STATUS) for _ in range(WINDING_UP_COUNT)],
        'date_of_case_status':                                  [randdate(min(randomDates[i], randomDates[-i - 1]), max(randomDates[i], randomDates[-i - 1])).strftime('%Y/%m/%d') for i in range(WINDING_UP_COUNT)],
        'other_party_status':                                   [random.choice(PARTY_STATUS) for _ in range(WINDING_UP_COUNT)],
        'date_of_other_party_status':                           [randdate(min(randomDates[i], randomDates[-i - 1]), max(randomDates[i], randomDates[-i - 1])).strftime('%Y/%m/%d') for i in range(WINDING_UP_COUNT)],
        'defendant_status':                                     [random.choice(PARTY_STATUS) for _ in range(WINDING_UP_COUNT)],
        'date_of_defendant_status':                             [randdate(min(randomDates[i], randomDates[-i - 1]), max(randomDates[i], randomDates[-i - 1])).strftime('%Y/%m/%d') for i in range(WINDING_UP_COUNT)],
        'amount':                                               [f'1) SGD {random.randint(100000, 50000000)}' for _ in range(WINDING_UP_COUNT)],
        'last_updated':                                         [max(randomDates[i], randomDates[-i - 1]).strftime('%Y/%m/%d') for i in range(WINDING_UP_COUNT)],
    },
    'OS OA OP OM':{
        'type':                                                 ['ORIGINATING SUMMONS' for _ in range(OSOAOPOM_COUNT)],
        'case_no':                                              [random.randint(1, 10000) for _ in range(OSOAOPOM_COUNT)],
        'year':                                                 [min(randomDates[i], randomDates[-i - 1]).year for i in range(len(randdates(2 * OSOAOPOM_COUNT, reg_date, req_date)) // 2)], 
        'date_of_filling':                                      [min(randomDates[i], randomDates[-i - 1]).strftime('%Y/%m/%d') for i in range(OSOAOPOM_COUNT)],
        'other_party_name':                                     [f'{random.choice(firstName)} {random.choice(middleName)} {random.choice(sector)} {random.choice(companyType)}' for _ in range(OSOAOPOM_COUNT)],
        'case_status':                                          [random.choice(CASE_STATUS) for _ in range(OSOAOPOM_COUNT)],
        'date_of_case_status':                                  [randdate(min(randomDates[i], randomDates[-i - 1]), max(randomDates[i], randomDates[-i - 1])).strftime('%Y/%m/%d') for i in range(OSOAOPOM_COUNT)],
        'other_party_status':                                   [random.choice(PARTY_STATUS) for _ in range(OSOAOPOM_COUNT)],
        'date_of_other_party_status':                           [randdate(min(randomDates[i], randomDates[-i - 1]), max(randomDates[i], randomDates[-i - 1])).strftime('%Y/%m/%d') for i in range(OSOAOPOM_COUNT)],
        'defendant_status':                                     [random.choice(PARTY_STATUS) for _ in range(OSOAOPOM_COUNT)],
        'date_of_defendant_status':                             [randdate(min(randomDates[i], randomDates[-i - 1]), max(randomDates[i], randomDates[-i - 1])).strftime('%Y/%m/%d') for i in range(OSOAOPOM_COUNT)],
        'amount':                                               [f'1) SGD {random.randint(100000, 50000000)}' for _ in range(OSOAOPOM_COUNT)],
        'last_updated':                                         [max(randomDates[i], randomDates[-i - 1]).strftime('%Y/%m/%d') for i in range(OSOAOPOM_COUNT)],
    },
    'Add Exp Prop. Info':{
        'num_of_employee':                                      random.randint(10000, 50000), 
        'num_of_commerical_properties_owned':                   random.choice(['-', random.randint(1, 100)]),
    },
    'Accreditations':{
        'award_name':                                           ['SINGAPORE 1000 COMPANIES RANKED BY SALES/TURNOVER' for _ in range(ACRRED_COUNT)], 
        'year_of_award':                                        [i.year for i in randdates(ACRRED_COUNT, reg_date, req_date)], 
        'award_ranking':                                        ['-' for _ in range(ACRRED_COUNT)],
    },
    'Financial Related Search Count':{
        'entity_id':                                            ENTITY_ID,
        'year':                                                 f'20XX',
        'month':                                                None,
        'purpose':                                              list(random.sample(FINANCIAL_REL_SEARCH_COUNT_VAL, random.randint(1, len(FINANCIAL_REL_SEARCH_COUNT_VAL)))),
        'count':                                                None,
    },
    'Commercial Related Search Count':{
        'entity_id':                                            ENTITY_ID,
        'year':                                                 f'20XX',
        'month':                                                None,
        'purpose':                                              list(random.sample(COMMERCIAL_REL_SEARCH_COUNT_VAL, random.randint(1, len(COMMERCIAL_REL_SEARCH_COUNT_VAL)))),
        'count':                                                None,
    },
    'Legal Related Search Count':{
        'entity_id':                                            ENTITY_ID,
        'year':                                                 f'20XX',
        'month':                                                None,
        'purpose':                                              list(random.sample(LEGAL_REL_SEARCH_COUNT_VAL, random.randint(1, len(LEGAL_REL_SEARCH_COUNT_VAL)))),
        'count':                                                None,
    },
}

In [5]:
def random_Related_Search_Count_Generation(purposes, entityId, year):
    # Generate the 'month' and 'year' column
    months = ['All', 'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    month_col = months * (len(purposes) + 3)
    year_col = [year] * len(months) * (len(purposes) + 1)
    year_col.extend([year-1] * len(months))
    year_col.extend([year-2] * len(months))

    # Generate the purpose column    
    purpose_col = ['All'] * 13 + list(itertools.chain(*[[ele] * 13 for ele in purposes])) + ['All'] * 26


    # Most recent 2 years section
    def random_row_generation():
        try:
            randomAll = random.randint(0, 5)
            num_of_splits = random.randint(0, randomAll)
            splits = []
            for i in range(num_of_splits - 1):
                split = random.randrange(1, randomAll - sum(splits))
                splits.append(split)
            splits.append(randomAll - sum(splits))

            # Create the data row
            row = [randomAll]
            monthsValue = [0] * (12 - len(splits)) + splits
            random.shuffle(monthsValue)
            row.extend(monthsValue)
            return row
        except ValueError:
            return [0] * 13

    randomRows = [random_row_generation() for i in range(len(purposes))]
    totalSum = [sum(row) for row in zip(*randomRows)]
    
    # Generate the Count Column
    count_col = totalSum + list(itertools.chain(*randomRows)) + random_row_generation() + random_row_generation()

    # Genrate the entity column
    entity_col = [entityId for i in range(len(count_col))]

    # Creation of Dataframe
    dataframe = pd.DataFrame(data= {
        'entity_id': entity_col,
        'year': year_col, 
        'month': month_col, 
        'purpose': purpose_col, 
        'count': count_col
    }, columns=['entity_id', 'year', 'month', 'purpose', 'count'])

    return dataframe

Importing the processed data for comparison

In [6]:
df_dict = pd.read_excel('../../../eins/Output/19940XXXXW.xlsx', sheet_name=None, usecols=lambda col: col != 'Unnamed: 0')

Table 1: Request Criteria

In [7]:
print('Given Input')
df_dict['Request Criteria']

Given Input


,request_date,requestor_name,requested_entity_name,requested_entity_id,request_id
0,01/06/20XX,XYZ INFORMATION PTE LTD,GREENYARD PIPE SYSTEMS PTE LTD,19940XXXXW,EINSG SAMPLE


In [8]:
print('Generated Output')
random_req_criteria = pd.DataFrame(
    data=[list(random_params['Request Criteria'].values())], columns=random_params['Request Criteria'].keys()
)
random_req_criteria

Generated Output


,request_date,requestor_name,requested_entity_name,requested_entity_id,request_id
0,2023/01/27,DUSK METAL COAST CORP,SUNSHINE PLASTIC STATE PTE LIMITED,200119826B,EINSG SAMPLE


Table 2: Search Record

In [9]:
print('Given Input')
df_dict['Search Record']

Given Input


,entity_name,entity_id
0,1) GREENYARD PIPE SYSTEMS PTE LTD\n2) GREENYAR...,19940XXXXW


In [10]:
print('Generated Output')
random_search_rec = pd.DataFrame(
    data=[list(random_params['Search Record'].values())], columns=random_params['Search Record'].keys()
)
random_search_rec

Generated Output


,entity_name,entity_id
0,SUNSHINE PLASTIC STATE PTE LIMITED,200119826B


Table 3: Details of Company

In [11]:
print('Given Input')
df_dict['Details of Company']

Given Input


,entity_name,entity_id,registration_date,country_or_region_of_incorporation_or_registration,date_of_name_change,former_name,effective_date_of_former_name,company_type,registered_office_address,date_of_address_change,principal_activities,status,status_date
0,GREENYARD PIPE SYSTEMS PTE LTD,19940XXXXW,12/05/19XX,SINGAPORE,01/10/19XX,GREENYARD PLASTICS INTERNATIONAL PTE LIMITED,12/05/1994,PRIVATE COMPANY LIMITED BY SHARES,99 CHANGI NORTH WAY\n#04-04\nSINGAPORE 490000,22/05/20XX,1) PLASTIC PRODUCTS SERVICES (27220),LIVE COMPANY,12/05/19XX


In [14]:
print('Generated Output')
random_details_of_comp = pd.DataFrame(
    data=[list(random_params['Details of Company'].values())], columns=random_params['Details of Company'].keys()
)
principal_activities = random_details_of_comp.loc[0, 'principal_activities']
ssic_number = re.findall(r'\((\d+)\)', principal_activities)[0]
random_details_of_comp = pd.concat([random_details_of_comp, ssic_mapping(ssic_number, ssic_mapper)], axis=1)
random_details_of_comp

Generated Output


,entity_name,entity_id,registration_date,country_or_region_of_incorporation_or_registration,date_of_name_change,former_name,effective_date_of_former_name,company_type,registered_office_address,date_of_address_change,principal_activities,status,status_date,Section,Division,Group,Class,Sub-Class
0,SUNSHINE PLASTIC STATE PTE LIMITED,200119826B,2001/02/09,MALAYSIA,2015/03/14,SUNSHINE PLASTIC STATE PTE LIMITED,2012/08/13,UNLIMITED EXEMPT PRIVATE COMPANY,JURONG AVENUE #18- 31 SINGAPORE 926724,2013/09/06,Retail sale of specific commodities n.e.c. (47...,GAZETTED TO BE STRUCK OFF,2009/05/23,WHOLESALE AND RETAIL TRADE,RETAIL TRADE,RETAIL SALE OF OTHER GOODS IN SPECIALISED STORES,Retail Sale of Other Goods in Specialised Stores,Retail sale of specific commodities n.e.c.


Table 4: Capital Structure

In [13]:
print('Given Input')
df_dict['Capital Structure']

Given Input


,issued_ordinary_no_of_shares,currency_of_issued_ordinary,amount_of_issued_ordinary,paid_up_ordinary_no_of_shares,currency_of_paid_up_ordinary,amount_of_paid_up_ordinary,paid_up_preference_no_of_shares,currency_of_paid_up_preference,amount_of_paid_up_preference,paid_up_others_no_of_shares,currency_of_paid_up_others,amount_of_paid_up_others
0,"9,661,441.00","SINGAPORE, DOLLARS","9,661,441.00",-,"SINGAPORE, DOLLARS","9,661,441.00",-,"SINGAPORE, DOLLARS",0.0,-,"SINGAPORE, DOLLARS",0.0


In [14]:
print('Generated Output')
random_cap_struct = pd.DataFrame(
    data=[list(random_params['Capital Structure'].values())], columns=random_params['Capital Structure'].keys()
)
random_cap_struct

Generated Output


,entity_id,issued_ordinary_no_of_shares,currency_of_issued_ordinary,amount_of_issued_ordinary,paid_up_ordinary_no_of_shares,currency_of_paid_up_ordinary,amount_of_paid_up_ordinary,paid_up_preference_no_of_shares,currency_of_paid_up_preference,amount_of_paid_up_preference,paid_up_others_no_of_shares,currency_of_paid_up_others,amount_of_paid_up_others
0,49218116F,3763635,"SINGAPORE, DOLLARS",1672608,4467009,"SINGAPORE, DOLLARS",7960454,6495890,"SINGAPORE, DOLLARS",4347952,130466,"SINGAPORE, DOLLARS",1718727


Table 5: Charges

In [15]:
print('Given Input')
df_dict['Charges']

Given Input


,charge_id,date_of_charge,chargee,amount_secured_and_currency,status
0,C2007XXXXX,16/01/20XX,BANK NAME,"400,000,000.00 SINGAPORE, DOLLARS",CURRENT
1,C2008XXXXX,10/02/20XX,BANK NAME,ALL MONIES,PARTIAL DISCHARGE AND REMAIN IN REGISTER
2,C2013XXXXX,16/01/20XX,BANK NAME,"97,000,000.00 SINGAPORE, DOLLARS",PARTIAL SATISFACTION AND REMAIN IN REGISTER


In [16]:
print('Generated Output')
random_charges = pd.DataFrame(data=random_params['Charges'], columns=random_params['Charges'])
random_charges

Generated Output


,charge_id,date_of_charge,chargee,amount_secured_and_currency,status,entity_id
0,C201840603,2018/06/22,BANK NAME,"9909512 SINGAPORE, DOLLARS",PARTIAL SATISFACTION AND REMAIN IN REGISTER,49218116F
1,C201669085,2016/07/09,BANK NAME,"27921812 SINGAPORE, DOLLARS",CURRENT,49218116F
2,C201364378,2013/10/01,BANK NAME,"41460902 SINGAPORE, DOLLARS",PARTIAL SATISFACTION AND REMAIN IN REGISTER,49218116F
3,C201757267,2017/09/07,BANK NAME,"99358961 SINGAPORE, DOLLARS",CURRENT,49218116F
4,C201327704,2013/08/12,BANK NAME,"23436761 SINGAPORE, DOLLARS",CURRENT,49218116F


Table 6: Auditors

In [17]:
print('Given Input')
df_dict['Auditors']

Given Input


,auditor_name,date_of_appointment
0,KPMG LLP,28/06/20XX


In [18]:
print('Generated Output')
random_auditors = pd.DataFrame(
    data=[list(random_params['Auditors'].values())], columns=random_params['Auditors'].keys()
)
random_auditors

Generated Output


,auditor_name,date_of_appointment,entity_id
0,EY,2014/12/28,49218116F


Table 7: Officers Auth. Rep

In [19]:
print('Given Input')
df_dict['Officers Auth. Rep']

Given Input


,employee_name,employee_id,employee_address,date_of_change_of_address,nationality_or_citizenship,date_of_appointment,position_held
0,GEORGIO SOUZ,XXX600M,"VIA FIASELLA 09/29 X-16000\nGENOVA,ITALY",-,ITALIAN,21/05/20XX,DIRECTOR
1,CHAN MEI MEI,GXXX1999U,"5X TAMPINES AVENUE 1\n#0X-0X TROPICA, THE\nSIN...",-,MALAYSIAN,20/02/20XX,SECRETARY
2,SHARVIL BIN JUMAAT,SXXX7999B,"1XX MILTONIA CLOSE\nSHAUGHNESSY, THE\nSINGAPOR...",19/10/20XX,SINGAPORE CITIZEN,21/05/20XX,DIRECTOR
3,LIN MEE TUN,SXXX6000J,4XX PASIR RIS DRIVE 6\n#0X-2XX\nSINGAPORE 510444,08/04/20XX,SINGAPORE P.R.,01/08/20XX,SECRETARY


In [20]:
print('Generated Output')
random_off_Auth_Rep = pd.DataFrame(
    data=random_params['Officers Auth. Rep'], columns=random_params['Officers Auth. Rep'].keys()
)
random_off_Auth_Rep

Generated Output


,employee_name,employee_id,employee_address,date_of_change_of_address,nationality_or_citizenship,date_of_appointment,position_held,entity_id
0,SUSAN NG,O0RRYU,PASIR RIS BROADWAY #23- 37 UK 233936,2021/04/08,MALAYSIA,2015/08/04,SECRETARY,49218116F
1,GOTHRY NG,9Q1N6N,PUNGGOL BROADWAY #12- 11 ITALY 331358,2020/08/30,MALAYSIA,2023/02/27,DIRECTOR,49218116F
2,KEVIN NG,AB4ZDM,CALDECOTT AVENUE #11- 6 US 789522,-,THAILAND,2022/03/06,DIRECTOR,49218116F
3,KEVIN BIN,DDTD3M,PASIR RIS BROADWAY #18- 32 VIETNAM 415745,-,MALAYSIA,2016/01/17,SECRETARY,49218116F
4,MABLE HOW,6Q72B1,MILTON HAWKWAY BOULEVARD #2- 14 VIETNAM 526907,-,SINGAPORE,2019/06/22,DIRECTOR,49218116F
5,TOM HOW,616SK9,JURONG MAISON #25- 40 VIETNAM 538571,-,US,2015/10/01,SECRETARY,49218116F


Table 8: Shareholders

In [21]:
print('Given Input')
df_dict['Shareholders']

Given Input


,shareholder_name,shareholder_id,nationality_or_citizenship,shareholder_address,date_of_change_of_address,type_of_shares,no_of_shares,currency
0,GREENYARD HOLDINGS LIMITED,19880XXXXG,SINGAPORE,3XX PASIR RIS DRIVE 1\n#0X-1XX\nSINGAPORE 110113,-,ORDINARY,"9,000,000.00","SINGAPORE, DOLLARS"
1,GREENYARD SINGPORE PTE. LTD.,20000XXXXG,SINGAPORE,1XX PASIR RIS DRIVE 2\n#0X-1XX\nSINGAPORE 110111,-,ORDINARY,"661,441.00","SINGAPORE, DOLLARS"


In [22]:
print('Generated Output')
random_shareholders = pd.DataFrame(
    data=random_params['Shareholders'], columns=random_params['Shareholders'].keys()
)
random_shareholders

Generated Output


,shareholder_name,shareholder_id,nationality_or_citizenship,shareholder_address,date_of_change_of_address,type_of_shares,no_of_shares,currency,entity_id
0,MIDNIGHT AIR INTERNATIONAL PTE LIMITED,U7H48Q7J,THAILAND,PUNGGOL BOULEVARD #10- 31 SINGAPORE 362017,-,REDEEMABLE PREFERENCE,79862692,"SINGAPORE, DOLLARS",49218116F


Table 9: Compliance Record

In [23]:
print('Given Input')
df_dict['Compliance Record']

Given Input


,date_of_last_agm,date_of_last_ar,date_of_ac_laid_at_last_agm
0,01/03/20XX,28/03/20XX,31/12/20XX


In [24]:
print('Generated Output')
random_compliance_rec = pd.DataFrame(
    data=[list(random_params['Compliance Record'].values())], columns=random_params['Compliance Record'].keys()
)
random_compliance_rec


Generated Output


,date_of_last_agm,date_of_last_ar,date_of_ac_laid_at_last_agm,entity_id
0,2019/03/16,2013/08/02,2021/04/03,49218116F


Table 10: Financial Data

In [25]:
print('Given Input')
df_dict['Financial Data']

Given Input


,current_fye,previous_fye,currency,current_fye_revenue,previous_fye_revenue,percentage_change_revenue,current_fye_profit_after_tax,previous_fye_profit_after_tax,percentage_change_profit_after_tax,current_fye_shareholders_fund,previous_fye_shareholders_fund,percentage_change_shareholders_fund
0,31/12/20XX,31/12/20XX,"SINGAPORE, DOLLARS","15,098,200,000","14,857,800,000",1.61,"441,600,000","396,800,000",11.29,"13,417,300,000","13,187,400,000",1.74


In [26]:
print('Generated Output')
random_fin_Data = pd.DataFrame(
    data=[list(random_params['Financial Data'].values())], columns=random_params['Financial Data'].keys()
)
random_fin_Data

Generated Output


,entity_id,current_fye,previous_fye,currency,current_fye_revenue,previous_fye_revenue,percentage_change_revenue,current_fye_profit_after_tax,previous_fye_profit_after_tax,percentage_change_profit_after_tax,current_fye_shareholders_fund,previous_fye_shareholders_fund,percentage_change_shareholders_fund
0,49218116F,31/12/2022,31/12/2021,"SINGAPORE, DOLLARS",31319061847,60849641064,-48.53,30329659864,60578667261,-49.93,30380946111,59902774416,-49.28


Table 11: Key Ratios

In [27]:
print('Given Input')
df_dict['Key Ratios']

Given Input


,current_fye,previous_fye,current_fye_profit_margin,previous_fye_profit_margin,current_fye_liquidity_times,previous_fye_liquidity_times,current_fye_debt_times,previous_fye_debt_times
0,31/12/20XX,31/12/20XX,2.92%,2.67%,1.16,1.2,0.71,1.1


In [28]:
print('Generated Output')
random_key_Ratio = pd.DataFrame(
    data=[list(random_params['Key Ratios'].values())], columns=random_params['Key Ratios'].keys()
)
random_key_Ratio

Generated Output


,entity_id,current_fye,previous_fye,current_fye_profit_margin,previous_fye_profit_margin,current_fye_liquidity_times,previous_fye_liquidity_times,current_fye_debt_times,previous_fye_debt_times
0,49218116F,31/12/2022,31/12/2021,3.61%,2.72%,1.83,1.71,2.32,1.27


Table 12: Analysis

In [29]:
print('Given Input')
df_dict['Analysis']

Given Input


,fye,company_level_revenue_growth,industry_median_revenue_growth,company_level_return_on_assets,industry_median_return_on_assets,company_level_total_asset_turnover,industry_median_total_asset_turnover,company_level_net_profit_margin,industry_median_net_profit_margin
0,31/12/20XX,28.14%,10.03%,1.82%,5.84%,27.77%,2.45%,2.92%,3.05%


In [30]:
print('Generated Output')
random_analysis = pd.DataFrame(
    data=[list(random_params['Analysis'].values())], columns=random_params['Analysis'].keys()
)
random_analysis

Generated Output


,entity_id,fye,company_level_revenue_growth,industry_median_revenue_growth,company_level_return_on_assets,industry_median_return_on_assets,company_level_total_asset_turnover,industry_median_total_asset_turnover,company_level_net_profit_margin,industry_median_net_profit_margin
0,49218116F,2022,59.95%,11.34%,4.59%,4.29%,3.28%,7.78%,2.09%,2.84%


Table 13: Writ of Sum. Org. Claim

In [31]:
print('Given Input')
df_dict['Writ of Sum. Org. Claim']

Given Input


,sn,court_type,case_no,year,date_of_filling,plaintiff,case_status,date_of_case_status,case_status_remarks,plaintiff_status,date_of_plaitiff_status,defendant_status,date_of_defendant_status,amount,last_updated,main_cause
0,1,SC/WOS,10,20XX,16/02/20XX,ABC PTE LTD,PENDING,20/04/20XX,NaN,PENDING,20/04/20XX,PENDING,20/04/20XX,"1) SGD 2,000,000.00\n2) SGD 50,000.00\n3) SGD ...",16/05/20XX,CONTRACTS - GENERAL
1,2,DC/WOS,97,20XX,17/07/20XX,SCC CAPITAL\nLTD,CONCLUDED,20/03/20XX,TRANSFERREDFROM MC5401/20XX,CONCLUDED,20/03/20XX,CONCLUDED,20/03/20XX,"1) SGD 50,000.00",20/04/20XX,GOOD SOLD AND DELIVERED
2,3,MC/OC,5401,20XX,09/06/20XX,SCC CAPITAL\nLTD,TRANSFERRED,17/07/20XX,TRANSFERRED TODC 97/20XX,-,NaN,-,NaN,-,15/01/20XX,GOOD SOLD AND DELIVERED


In [32]:
print('Generated Output')
random_writ_SOC = pd.DataFrame(
    data=random_params['Writ of Sum. Org. Claim'], columns=random_params['Writ of Sum. Org. Claim'].keys()
)
random_writ_SOC['main_cause_category'] = random_writ_SOC['main_cause'].apply(lambda x: label_litigation_main_cause(litigation_main_cause_clustering_model, x))
random_writ_SOC['case_status_category'] = random_writ_SOC['case_status'].apply(lambda x: label_litigation_case_status(litigation_case_status_clustering_model, x))
random_writ_SOC['plaintiff_status_category'] = random_writ_SOC['plaintiff_status'].apply(lambda x: label_litigation_party_status(litigation_party_status_clustering_model, x))
random_writ_SOC['defendant_status_category'] = random_writ_SOC['defendant_status'].apply(lambda x: label_litigation_party_status(litigation_party_status_clustering_model, x))
random_writ_SOC

Generated Output


,sn,court_type,case_no,year,date_of_filling,plaintiff,case_status,date_of_case_status,case_status_remarks,plaintiff_status,...,defendant_status,date_of_defendant_status,amount,last_updated,main_cause,entity_id,main_cause_category,case_status_category,plaintiff_status_category,defendant_status_category
0,1,MC / WOS,7223,2017,2017/08/08,DUSK WATER STATE ENTERPRISE,CONCLUDED (31-JUL-2013),2018/02/05,BO ANNULLED/DISCHARGED,ORDER INTERMS,...,BANKRUPTY ORDER,2020/09/23,1) SGD 18565000,2021/05/25,Contract-Others(Investment Agreement dated 22 ...,49218116F,Contracts,Concluded,Others,Bankruptcy Related


Table 14: Winding Up

In [33]:
print('Given Input')
df_dict['Winding Up']

Given Input


,type,case_no,year,date_of_filling,other_party_name,case_status,date_of_case_status,other_party_status,date_of_other_party_status,defendant_status,date_of_defendant_status,amount,last_updated
0,WINDING UP,123,20XX,02/06/20XX,ABC & CO,CONCLUDED,02/09/20XX,DISCONTINUED/WITHDRAWN,02/09/20XX,CONCLUDED,02/09/20XX,"SGD 2,000,000.00",07/11/20XX


In [34]:
print('Generated Output')
random_winding_Up = pd.DataFrame(
    data=random_params['Winding Up'], columns=random_params['Winding Up'].keys()
)
random_winding_Up['case_status_category'] = random_winding_Up['case_status'].apply(lambda x: label_litigation_case_status(litigation_case_status_clustering_model, x))
random_winding_Up['other_party_status_category'] = random_winding_Up['other_party_status'].apply(lambda x: label_litigation_party_status(litigation_party_status_clustering_model, x))
random_winding_Up['defendant_status_category'] = random_winding_Up['defendant_status'].apply(lambda x: label_litigation_party_status(litigation_party_status_clustering_model, x))
random_winding_Up

Generated Output


,type,case_no,year,date_of_filling,other_party_name,case_status,date_of_case_status,other_party_status,date_of_other_party_status,defendant_status,date_of_defendant_status,amount,last_updated,entity_id,case_status_category,other_party_status_category,defendant_status_category
0,WINDING UP,2853,2015,2015/04/04,TWILIGHT WOOD STATE PTE LIMITED,NOTICE OF DISCHARGE UNDER SECTION 123 (3),2018/09/17,NOTICE OF DISCHARGE UNDER SECTION 95A,2017/08/08,ANNULLED,2020/11/17,1) SGD 18981939,2021/09/05,49218116F,Discharged,Discharge Notice,Annulment
1,WINDING UP,10000,2018,2018/07/17,DAWN METAL REGIONAL PTE LIMITED,TRANSFERRED,2019/10/27,BO RESCINDED,2020/09/10,NOTICEC OF DISCHARGE UNDER SECTION 125,2019/08/08,1) SGD 21338226,2020/10/30,49218116F,Others,Others,Discharge Notice
2,WINDING UP,2589,2014,2014/03/17,SUNSHINE GREENYARN CENTRAL PTE LIMITED,PETITION FILED,2022/06/10,PENDING - JUDICIAL MANAGEMENT,2019/05/02,"LIN HUI YIN,SHARON",2021/09/11,1) SGD 5649278,2022/09/27,49218116F,Others,Judgement,Others
3,WINDING UP,1405,2017,2017/02/14,DUSK WOOD REGIONAL PTE LIMITED,CONCLUDED (24-JAN-2011),2018/10/07,DEFAULT JUDGMENT-O13(12-NOV-2013),2019/09/25,CHOA CHU KANG CENTRAL,2019/09/09,1) SGD 11600967,2020/09/30,49218116F,Concluded,Judgement,Others


Table 15: OS OA OP OM

In [35]:
print('Given Input')
df_dict['OS OA OP OM']

Given Input


,type,case_no,year,date_of_filing,other_party_name,case_status,date_of_case_status,other_party_status,date_of_other_party_status,defendant_status,date_of_defendant_status,amount,last_updated
0,ORIGINATING\nSUMMONS,56,20XX,15/02/20XX,XYZ & CO,CONCLUDED,11/12/20XX,CONCLUDED,11/12/20XX,CONCLUDED,11/12/20XX,"SGD 2,000,000.00",07/11/20XX


In [36]:
print('Generated Output')
random_osoaopom = pd.DataFrame(
    data=random_params['OS OA OP OM'], columns=random_params['OS OA OP OM'].keys()
)
random_osoaopom['case_status_category'] = random_osoaopom['case_status'].apply(lambda x: label_litigation_case_status(litigation_case_status_clustering_model, x))
random_osoaopom['other_party_status_category'] = random_osoaopom['other_party_status'].apply(lambda x: label_litigation_party_status(litigation_party_status_clustering_model, x))
random_osoaopom['defendant_status_category'] = random_osoaopom['defendant_status'].apply(lambda x: label_litigation_party_status(litigation_party_status_clustering_model, x))
random_osoaopom

Generated Output


,type,case_no,year,date_of_filling,other_party_name,case_status,date_of_case_status,other_party_status,date_of_other_party_status,defendant_status,date_of_defendant_status,amount,last_updated,entity_id,case_status_category,other_party_status_category,defendant_status_category
0,ORIGINATING SUMMONS,6744,2017,2017/12/27,MOONLIGHT RIVER COAST ENTERPRISE,APPLICATION FOR BANKRUPTCY APPLICATION,2019/02/07,BANKRUPTCY ORDERS,2020/03/09,"SETTLED -AFTER SET DOWN, BEFORE TRIAL",2017/12/31,1) SGD 43193297,2020/08/23,49218116F,Bankruptcy Related,Bankruptcy Related,Judgement
1,ORIGINATING SUMMONS,6262,2022,2022/04/30,MIDNIGHT WOOD COAST CORP,ORDER MADE ON COMPULSORY WINDING UP,2022/05/27,ORDERS MADE ON 04 JUL 1997 AGAINST THE BANKRUP...,2022/06/02,FINAL JUDGMENT WITHOUT TRIAL,2022/05/25,1) SGD 39979882,2022/07/28,49218116F,Bankruptcy Related,Bankruptcy Related,Judgement


Table 16: Add Exp Prop. Info

In [37]:
print('Given Input')
df_dict['Add Exp Prop. Info']

Given Input


,num_of_employee,num_of_commerical_properties_owned
0,31834,-


In [38]:
print('Generated Output')
random_add_Exp_Prop_Info =pd.DataFrame(
    data=[list(random_params['Add Exp Prop. Info'].values())], columns=random_params['Add Exp Prop. Info'].keys()
)
random_add_Exp_Prop_Info

Generated Output


,entity_id,num_of_employee,num_of_commerical_properties_owned
0,49218116F,35159,23


Table 17: Accreditations

In [39]:
print('Given Input')
df_dict['Accreditations']

Given Input


,award_name,year_of_award,award_ranking
0,SINGAPORE 1000 COMPANIES RANKED BY SALES/TURNOVER,20XX,XXX
1,SINGAPORE 1000 COMPANIES RANKED BY SALES/TURNOVER,20XX,XXX


In [40]:
print('Generated Output')
random_accreditation = pd.DataFrame(
    data=random_params['Accreditations'], columns=random_params['Accreditations'].keys()
)
random_accreditation

Generated Output


,entity_id,award_name,year_of_award,award_ranking
0,49218116F,SINGAPORE 1000 COMPANIES RANKED BY SALES/TURNOVER,2019,-
1,49218116F,SINGAPORE 1000 COMPANIES RANKED BY SALES/TURNOVER,2013,-


Table 18: Financial Related Search Count

In [41]:
print('Given Input')
df_dict['Financial Related Search Count']

Given Input


,year,month,purpose,count
0,20XX,All,All,3
1,20XX,Jan,All,3
2,20XX,Feb,All,0
3,20XX,Mar,All,0
4,20XX,Apr,All,0
...,...,...,...,...
58,20XX,Aug,All,0
59,20XX,Sep,All,0
60,20XX,Oct,All,0
61,20XX,Nov,All,0


In [42]:
print('Generated Output')
random_fin_search_count = random_Related_Search_Count_Generation(random_params['Financial Related Search Count']['purpose'],
        random_params['Financial Related Search Count']['entity_id'], req_date.year-1
    )
random_fin_search_count

Generated Output


,entity_id,year,month,purpose,count
0,49218116F,2022,All,All,10
1,49218116F,2022,Jan,All,2
2,49218116F,2022,Feb,All,0
3,49218116F,2022,Mar,All,0
4,49218116F,2022,Apr,All,0
...,...,...,...,...,...
86,49218116F,2020,Aug,All,0
87,49218116F,2020,Sep,All,0
88,49218116F,2020,Oct,All,0
89,49218116F,2020,Nov,All,0


Table 19: Commercial Related Search Count

In [43]:
print('Given Input')
df_dict['Commercial Related Search Count']

Given Input


,year,month,purpose,count
0,20XX,All,All,1
1,20XX,Jan,All,1
2,20XX,Feb,All,0
3,20XX,Mar,All,0
4,20XX,Apr,All,0
5,20XX,May,All,0
6,20XX,Jun,All,0
7,20XX,Jul,All,0
8,20XX,Aug,All,0
9,20XX,Sep,All,0


In [44]:

random_comm_rel_search_count = random_Related_Search_Count_Generation(random_params['Commercial Related Search Count']['purpose'],
        random_params['Commercial Related Search Count']['entity_id'], req_date.year-1
    )
random_comm_rel_search_count

,entity_id,year,month,purpose,count
0,49218116F,2022,All,All,1
1,49218116F,2022,Jan,All,0
2,49218116F,2022,Feb,All,0
3,49218116F,2022,Mar,All,0
4,49218116F,2022,Apr,All,0
5,49218116F,2022,May,All,0
6,49218116F,2022,Jun,All,0
7,49218116F,2022,Jul,All,0
8,49218116F,2022,Aug,All,0
9,49218116F,2022,Sep,All,0


Table 20: Legal Related Search Count 

In [45]:
print('Given Input')
df_dict['Legal Related Search Count']

Given Input


,year,month,purpose,count
0,20XX,All,All,2
1,20XX,Jan,All,2
2,20XX,Feb,All,0
3,20XX,Mar,All,0
4,20XX,Apr,All,0
...,...,...,...,...
58,20XX,Aug,All,0
59,20XX,Sep,All,0
60,20XX,Oct,All,0
61,20XX,Nov,All,0


In [46]:
random_legal_rel_search_count = random_Related_Search_Count_Generation(random_params['Legal Related Search Count']['purpose'],
        random_params['Legal Related Search Count']['entity_id'], req_date.year-1
    )
random_legal_rel_search_count

,entity_id,year,month,purpose,count
0,49218116F,2022,All,All,2
1,49218116F,2022,Jan,All,0
2,49218116F,2022,Feb,All,0
3,49218116F,2022,Mar,All,0
4,49218116F,2022,Apr,All,0
...,...,...,...,...,...
60,49218116F,2020,Aug,All,0
61,49218116F,2020,Sep,All,4
62,49218116F,2020,Oct,All,0
63,49218116F,2020,Nov,All,0


Export the randomly generated data

In [47]:
# Output Data
writer = pd.ExcelWriter(f'../../Outputs/EINS_Random_Output/{ENTITY_ID}.xlsx', engine='xlsxwriter')

random_req_criteria.to_excel(writer, sheet_name='Request Criteria')
random_search_rec.to_excel(writer, sheet_name='Search Record')
random_details_of_comp.to_excel(writer, sheet_name='Details of Company')
random_cap_struct.to_excel(writer, sheet_name='Capital Structure')
random_charges.to_excel(writer, sheet_name='Charges')
random_auditors.to_excel(writer, sheet_name='Auditors')
random_off_Auth_Rep.to_excel(writer, sheet_name='Officers Auth. Rep')
random_shareholders.to_excel(writer, sheet_name='Shareholders')
random_compliance_rec.to_excel(writer, sheet_name='Compliance Record')
random_fin_Data.to_excel(writer, sheet_name='Financial Data')
random_key_Ratio.to_excel(writer, sheet_name='Key Ratios')
random_analysis.to_excel(writer, sheet_name='Analysis')
random_writ_SOC.to_excel(writer, sheet_name='Writ of Sum. Org. Claim')
random_winding_Up.to_excel(writer, sheet_name='Winding Up')
random_osoaopom.to_excel(writer, sheet_name='OS OA OP OM')
random_add_Exp_Prop_Info.to_excel(writer, sheet_name='Add Exp Prop. Info')
random_accreditation.to_excel(writer, sheet_name='Accreditations')
random_fin_search_count.to_excel(writer, sheet_name='Financial Related Search Count')
random_comm_rel_search_count.to_excel(writer, sheet_name='Commercial Related Search Count')
random_legal_rel_search_count.to_excel(writer, sheet_name='Legal Related Search Count')

writer.close()

Merge the excel files together

In [48]:
# isinstance(random_req_criteria, pd.DataFrame) returns True
merged_req_criteria_data = pd.DataFrame()
merged_search_rec_data = pd.DataFrame()
merged_details_of_comp_data = pd.DataFrame()
merged_cap_struct_data = pd.DataFrame()
merged_charges_data = pd.DataFrame()
merged_auditors_data = pd.DataFrame()
merged_off_Auth_Rep_data = pd.DataFrame()
merged_shareholders_data = pd.DataFrame()
merged_compliance_rec_data = pd.DataFrame()
merged_fin_Data_data = pd.DataFrame()
merged_key_Ratio_data = pd.DataFrame()
merged_analysis_data = pd.DataFrame()
merged_writ_SOC_data = pd.DataFrame()
merged_winding_Up_data = pd.DataFrame()
merged_osoaopom_data = pd.DataFrame()
merged_add_Exp_Prop_Info_data = pd.DataFrame()
merged_accreditation_data = pd.DataFrame()
merged_fin_search_count_data = pd.DataFrame()
merged_comm_rel_search_count_data = pd.DataFrame()
merged_legal_rel_search_count_data = pd.DataFrame()

In [49]:
def has_special_character(filename):
    # defining set of allowed characters
    allowed_chars = set(string.ascii_letters + string.digits + '_.')

    # Check if each character in the filename is an allowed character
    for char in filename:
        if char not in allowed_chars:
            return True
    return False

In [50]:
directory_path = '../../Outputs/EINS_Random_Output'

for filename in os.listdir(directory_path):
    if filename.endswith('.xlsx') and filename != 'compiled_EINS_reports.xlsx' and not has_special_character(filename):
        req_criteria_sheet = pd.read_excel(os.path.join(directory_path, filename), sheet_name='Request Criteria', engine='openpyxl')
        req_search_rec_sheet = pd.read_excel(os.path.join(directory_path, filename), sheet_name='Search Record', engine='openpyxl')
        req_details_of_comp_sheet = pd.read_excel(os.path.join(directory_path, filename), sheet_name='Details of Company', engine='openpyxl')
        req_cap_struct_sheet = pd.read_excel(os.path.join(directory_path, filename), sheet_name='Capital Structure', engine='openpyxl')
        req_charges_sheet = pd.read_excel(os.path.join(directory_path, filename), sheet_name='Charges', engine='openpyxl')
        req_auditors_sheet = pd.read_excel(os.path.join(directory_path, filename), sheet_name='Auditors', engine='openpyxl')
        req_off_Auth_Rep_sheet = pd.read_excel(os.path.join(directory_path, filename), sheet_name='Officers Auth. Rep', engine='openpyxl')
        req_shareholders_sheet = pd.read_excel(os.path.join(directory_path, filename), sheet_name='Shareholders', engine='openpyxl')
        req_compliance_rec_sheet = pd.read_excel(os.path.join(directory_path, filename), sheet_name='Compliance Record', engine='openpyxl')
        req_fin_Data_sheet = pd.read_excel(os.path.join(directory_path, filename), sheet_name='Financial Data', engine='openpyxl')
        req_key_Ratio_sheet = pd.read_excel(os.path.join(directory_path, filename), sheet_name='Key Ratios', engine='openpyxl')
        req_analysis_sheet = pd.read_excel(os.path.join(directory_path, filename), sheet_name='Analysis', engine='openpyxl')
        req_writ_SOC_sheet = pd.read_excel(os.path.join(directory_path, filename), sheet_name='Writ of Sum. Org. Claim', engine='openpyxl')
        req_winding_Up_sheet = pd.read_excel(os.path.join(directory_path, filename), sheet_name='Winding Up', engine='openpyxl')
        req_osoaopom_sheet = pd.read_excel(os.path.join(directory_path, filename), sheet_name='OS OA OP OM', engine='openpyxl')
        req_add_Exp_Prop_Info_sheet = pd.read_excel(os.path.join(directory_path, filename), sheet_name='Add Exp Prop. Info', engine='openpyxl')
        req_accreditation_sheet = pd.read_excel(os.path.join(directory_path, filename), sheet_name='Accreditations', engine='openpyxl')
        req_fin_search_count_sheet = pd.read_excel(os.path.join(directory_path, filename), sheet_name='Financial Related Search Count', engine='openpyxl')
        req_comm_rel_search_count_sheet = pd.read_excel(os.path.join(directory_path, filename), sheet_name='Commercial Related Search Count', engine='openpyxl')
        req_legal_rel_search_count_sheet = pd.read_excel(os.path.join(directory_path, filename), sheet_name='Legal Related Search Count', engine='openpyxl')
        if len(merged_req_criteria_data) == 0: # first iteration
            merged_req_criteria_data = req_criteria_sheet.copy()
        else:
            merged_req_criteria_data = pd.concat([merged_req_criteria_data, req_criteria_sheet], ignore_index=True)
        if len(merged_search_rec_data) == 0: # first iteration
            merged_search_rec_data = req_search_rec_sheet.copy()
        else:
            merged_search_rec_data = pd.concat([merged_search_rec_data, req_search_rec_sheet], ignore_index=True)
        if len(merged_details_of_comp_data) == 0: # first iteration
            merged_details_of_comp_data = req_details_of_comp_sheet.copy()
        else:
            merged_details_of_comp_data = pd.concat([merged_details_of_comp_data, req_details_of_comp_sheet], ignore_index=True)
        if len(merged_cap_struct_data) == 0: # first iteration
            merged_cap_struct_data = req_cap_struct_sheet.copy()
        else:
            merged_cap_struct_data = pd.concat([merged_cap_struct_data, req_cap_struct_sheet], ignore_index=True)
        if len(merged_charges_data) == 0: # first iteration
            merged_charges_data = req_charges_sheet.copy()
        else:
            merged_charges_data = pd.concat([merged_charges_data, req_charges_sheet], ignore_index=True)
        if len(merged_auditors_data) == 0: # first iteration
            merged_auditors_data = req_auditors_sheet.copy()
        else:
            merged_auditors_data = pd.concat([merged_auditors_data, req_auditors_sheet], ignore_index=True)
        if len(merged_off_Auth_Rep_data) == 0: # first iteration
            merged_off_Auth_Rep_data = req_off_Auth_Rep_sheet.copy()
        else:
            merged_off_Auth_Rep_data = pd.concat([merged_off_Auth_Rep_data, req_off_Auth_Rep_sheet], ignore_index=True)
        if len(merged_shareholders_data) == 0: # first iteration
            merged_shareholders_data = req_shareholders_sheet.copy()
        else:
            merged_shareholders_data = pd.concat([merged_shareholders_data, req_shareholders_sheet], ignore_index=True)
        if len(merged_compliance_rec_data) == 0: # first iteration
            merged_compliance_rec_data = req_compliance_rec_sheet.copy()
        else:
            merged_compliance_rec_data = pd.concat([merged_compliance_rec_data, req_compliance_rec_sheet], ignore_index=True)
        if len(merged_fin_Data_data) == 0: # first iteration
            merged_fin_Data_data = req_fin_Data_sheet.copy()
        else:
            merged_fin_Data_data = pd.concat([merged_fin_Data_data, req_fin_Data_sheet], ignore_index=True)
        if len(merged_key_Ratio_data) == 0: # first iteration
            merged_key_Ratio_data = req_key_Ratio_sheet.copy()
        else:
            merged_key_Ratio_data = pd.concat([merged_key_Ratio_data, req_key_Ratio_sheet], ignore_index=True)
        if len(merged_analysis_data) == 0: # first iteration
            merged_analysis_data = req_analysis_sheet.copy()
        else:
            merged_analysis_data = pd.concat([merged_analysis_data, req_analysis_sheet], ignore_index=True)
        if len(merged_writ_SOC_data) == 0: # first iteration
            merged_writ_SOC_data = req_writ_SOC_sheet.copy()
        else:
            merged_writ_SOC_data = pd.concat([merged_writ_SOC_data, req_writ_SOC_sheet], ignore_index=True)
        if len(merged_winding_Up_data) == 0: # first iteration
            merged_winding_Up_data = req_winding_Up_sheet.copy()
        else:
            merged_winding_Up_data = pd.concat([merged_winding_Up_data, req_winding_Up_sheet], ignore_index=True)
        if len(merged_osoaopom_data) == 0: # first iteration
            merged_osoaopom_data = req_osoaopom_sheet.copy()
        else:
            merged_osoaopom_data = pd.concat([merged_osoaopom_data, req_osoaopom_sheet], ignore_index=True)
        if len(merged_add_Exp_Prop_Info_data) == 0: # first iteration
            merged_add_Exp_Prop_Info_data = req_add_Exp_Prop_Info_sheet.copy()
        else:
            merged_add_Exp_Prop_Info_data = pd.concat([merged_add_Exp_Prop_Info_data, req_add_Exp_Prop_Info_sheet], ignore_index=True)
        if len(merged_accreditation_data) == 0: # first iteration
            merged_accreditation_data = req_accreditation_sheet.copy()
        else:
            merged_accreditation_data = pd.concat([merged_accreditation_data, req_accreditation_sheet], ignore_index=True)
        if len(merged_fin_search_count_data) == 0: # first iteration
            merged_fin_search_count_data = req_fin_search_count_sheet.copy()
        else:
            merged_fin_search_count_data = pd.concat([merged_fin_search_count_data, req_fin_search_count_sheet], ignore_index=True)
        if len(merged_comm_rel_search_count_data) == 0: # first iteration
            merged_comm_rel_search_count_data = req_comm_rel_search_count_sheet.copy()
        else:
            merged_comm_rel_search_count_data = pd.concat([merged_comm_rel_search_count_data, req_comm_rel_search_count_sheet], ignore_index=True)
        if len(merged_legal_rel_search_count_data) == 0: # first iteration
            merged_legal_rel_search_count_data = req_legal_rel_search_count_sheet.copy()
        else:
            merged_legal_rel_search_count_data = pd.concat([merged_legal_rel_search_count_data, req_legal_rel_search_count_sheet], ignore_index=True)
        
        # included a special char at the front of filename to indicate that the file has already been compiled
        old_name = directory_path + '/' + filename
        new_name = directory_path + '/@' + filename
        os.rename(old_name, new_name)

# print(merged_req_criteria_data)

In [51]:
writer = pd.ExcelWriter('../../Outputs/EINS_Random_Output/compiled_EINS_reports.xlsx', engine='xlsxwriter')

merged_req_criteria_data.to_excel(writer, sheet_name='Request Criteria')
merged_search_rec_data.to_excel(writer, sheet_name='Search Record')
merged_details_of_comp_data.to_excel(writer, sheet_name='Details of Company')
merged_cap_struct_data.to_excel(writer, sheet_name='Capital Structure')
merged_charges_data.to_excel(writer, sheet_name='Charges')
merged_auditors_data.to_excel(writer, sheet_name='Auditors')
merged_off_Auth_Rep_data.to_excel(writer, sheet_name='Officers Auth. Rep')
merged_shareholders_data.to_excel(writer, sheet_name='Shareholders')
merged_compliance_rec_data.to_excel(writer, sheet_name='Compliance Record')
merged_fin_Data_data.to_excel(writer, sheet_name='Financial Data')
merged_key_Ratio_data.to_excel(writer, sheet_name='Key Ratios')
merged_analysis_data.to_excel(writer, sheet_name='Analysis')
merged_writ_SOC_data.to_excel(writer, sheet_name='Writ of Sum. Org. Claim')
merged_winding_Up_data.to_excel(writer, sheet_name='Winding Up')
merged_osoaopom_data.to_excel(writer, sheet_name='OS OA OP OM')
merged_add_Exp_Prop_Info_data.to_excel(writer, sheet_name='Add Exp Prop. Info')
merged_accreditation_data.to_excel(writer, sheet_name='Accreditations')
merged_fin_search_count_data.to_excel(writer, sheet_name='Financial Related Search Count')
merged_comm_rel_search_count_data.to_excel(writer, sheet_name='Commercial Related Search Count')
merged_legal_rel_search_count_data.to_excel(writer, sheet_name='Legal Related Search Count')

writer.close()